In [9]:
import cbbpy.mens_scraper as s
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd

def retrieve_game(game_id: int):
    """
    Returns boxscore_df and info_df for a single game.
    """
    info_df, boxscore_df, _= s.get_game(game_id=game_id, pbp=False, info=True)
    return boxscore_df, info_df

def compute_possessions(fga, fta, oreb, to):
    """Estimate possessions for a team."""
    return fga - oreb + to + (0.44 * fta)

def scrape_data(days_ago=1):
    """
    Scrapes all games from a given day.
    Returns two lists: boxscores and info dataframes.
    """
    date = datetime.today() - timedelta(days=days_ago)
    game_ids = s.get_game_ids(date)

    if not game_ids:
        print(f"No games found on {date.strftime('%Y-%m-%d')}")
        return [], []

    print(f"Found {len(game_ids)} game(s) on {date.strftime('%Y-%m-%d')}")

    boxscores = []
    info_list = []

    with ThreadPoolExecutor(max_workers=5) as executor:
        future_to_gid = {executor.submit(retrieve_game, gid): gid for gid in game_ids}
        for i, future in enumerate(as_completed(future_to_gid), start=1):
            gid = future_to_gid[future]
            try:
                boxscore_df, info_df = future.result()
                boxscores.append(boxscore_df)
                info_list.append(info_df)
                print(f"Retrieved {gid} ({i}/{len(game_ids)})")
            except Exception as e:
                print(f"Failed to retrieve {gid}: {e}")

    print("All games scraped!")
    boxscores = pd.concat(boxscores, ignore_index=True)
    info_list = pd.concat(info_list, ignore_index=True)

    return boxscores, info_list

def build_matchup_table(boxscores, info_list):
    """
    Combine boxscore and info_df to create matchup table with home/away/neutral.
    """
    # 
    boxscores = boxscores[boxscores['player'] == 'TEAM']
    cols = ['game_id', 'team', 'pts', 'fga', 'fta', 'oreb', 'to']
    boxscores = boxscores[cols]

    # Each game has two rows — split into "team" and "opponent" by pairing them
    df1 = boxscores.iloc[::2].reset_index(drop=True)   # first team in each game
    df2 = boxscores.iloc[1::2].reset_index(drop=True)  # second team in each game

    # Join side by side
    merged = pd.concat([
        df1.rename(columns=lambda c: c if c == 'game_id' else f'team_{c}'),
        df2.rename(columns=lambda c: f'opp_{c}' if c != 'game_id' else c).drop(columns='game_id')
    ], axis=1)

    # Create inverse rows (swap team/opp)
    inverse = merged.rename(columns=lambda c: c.replace('team_', '__tmp__').replace('opp_', 'team_').replace('__tmp__', 'opp_'))

    # Stack original + inverse
    result = pd.concat([merged, inverse], ignore_index=True).sort_values('game_id').reset_index(drop=True)

    result = result.merge(info_list[['game_id', 'home_team', 'is_neutral', 'game_day']], on = 'game_id')
    result = result.rename(columns={'team_team': 'team'})
    
    result['neutral'] = result['is_neutral'] == 1
    result['home'] = (~result['neutral']) & (result['home_team'] == result['team'])
    result['away'] = (~result['neutral']) & (result['home_team'] != result['team'])

    result['possession_team'] = compute_possessions(result['team_fga'], result['team_fta'], result['team_oreb'], result['team_to'])
    result['possession_opp'] = compute_possessions(result['opp_fga'], result['opp_fta'], result['opp_oreb'], result['opp_to'])

    result['ppp_off_team'] = result['team_pts']/ result['possession_team']
    result['ppp_def_team'] = result['opp_pts'] / result['possession_opp']

    result['ppp_off_opp'] = result['opp_pts'] / result['possession_opp'] 
    result['ppp_def_opp'] = result['team_pts'] / result['possession_team']

    return result


# Example usage in a Jupyter notebook
boxscores, info_list = scrape_data(days_ago=1)
print(info_list)
matchup_df = build_matchup_table(boxscores, info_list)
matchup_df

Found 23 game(s) on 2026-03-01
Retrieved 401828254 (1/23)
Retrieved 401825552 (2/23)
Retrieved 401828250 (3/23)
Retrieved 401828454 (4/23)
Retrieved 401825550 (5/23)
Retrieved 401825551 (6/23)
Retrieved 401813372 (7/23)
Retrieved 401830316 (8/23)
Retrieved 401830221 (9/23)
Retrieved 401828253 (10/23)
Retrieved 401828252 (11/23)
Retrieved 401813433 (12/23)
Retrieved 401813396 (13/23)
Retrieved 401828251 (14/23)
Retrieved 401813419 (15/23)
Retrieved 401813344 (16/23)
Retrieved 401813308 (17/23)
Retrieved 401822966 (18/23)
Retrieved 401830317 (19/23)
Retrieved 401858741 (20/23)
Retrieved 401830217 (21/23)
Retrieved 401812088 (22/23)
Retrieved 401828255 (23/23)
All games scraped!
      game_id game_status                 home_team home_id  home_rank  \
0   401828254       Final       South Florida Bulls      58        NaN   
1   401825552       Final       Ohio State Buckeyes     194        NaN   
2   401828250       Final               UAB Blazers       5        NaN   
3   401828454      

,game_id,team,team_pts,team_fga,team_fta,team_oreb,team_to,opp_team,opp_pts,opp_fga,...,game_day,neutral,home,away,possession_team,possession_opp,ppp_off_team,ppp_def_team,ppp_off_opp,ppp_def_opp
0,401812088,UNC Wilmington Seahawks,76,62,16,21,15,Charleston Cougars,79,56,...,"March 01, 2026",False,True,False,63.04,64.32,1.205584,1.228234,1.228234,1.205584
1,401812088,Charleston Cougars,79,56,28,10,6,UNC Wilmington Seahawks,76,62,...,"March 01, 2026",False,False,True,64.32,63.04,1.228234,1.205584,1.205584,1.228234
2,401813308,Manhattan Jaspers,65,64,19,10,12,Iona Gaels,69,55,...,"March 01, 2026",False,True,False,74.36,74.12,0.874126,0.930923,0.930923,0.874126
3,401813308,Iona Gaels,69,55,23,14,23,Manhattan Jaspers,65,64,...,"March 01, 2026",False,False,True,74.12,74.36,0.930923,0.874126,0.874126,0.930923
4,401813344,Saint Peter's Peacocks,63,49,15,10,17,Marist Red Foxes,56,63,...,"March 01, 2026",False,False,True,62.60,64.28,1.006390,0.871189,0.871189,1.006390
5,401813344,Marist Red Foxes,56,63,12,16,12,Saint Peter's Peacocks,63,49,...,"March 01, 2026",False,True,False,64.28,62.60,0.871189,1.006390,1.006390,0.871189
6,401813372,Quinnipiac Bobcats,67,59,30,14,7,Canisius Golden Griffins,63,60,...,"March 01, 2026",False,False,True,65.20,66.04,1.027607,0.953967,0.953967,1.027607
7,401813372,Canisius Golden Griffins,63,60,16,8,7,Quinnipiac Bobcats,67,59,...,"March 01, 2026",False,True,False,66.04,65.20,0.953967,1.027607,1.027607,0.953967
8,401813396,Niagara Purple Eagles,66,44,19,8,16,Merrimack Warriors,73,53,...,"March 01, 2026",False,True,False,60.36,59.72,1.093439,1.222371,1.222371,1.093439
9,401813396,Merrimack Warriors,73,53,13,10,11,Niagara Purple Eagles,66,44,...,"March 01, 2026",False,False,True,59.72,60.36,1.222371,1.093439,1.093439,1.222371


In [ ]:
import pandas as pd
from datetime import datetime, timedelta

def build_full_season(start_date, end_date):

    all_games = []

    total_days = (end_date - start_date).days

    for days_ago in range(total_days, -1, -1):

        target_date = datetime.today() - timedelta(days=days_ago)
        date_str = target_date.strftime("%Y-%m-%d")

        try:

            boxscores, info_list = scrape_data(days_ago=days_ago)

            matchup_df = build_matchup_table(
                boxscores,
                info_list,
            )

            if not matchup_df.empty:
                all_games.append(matchup_df)

            print(f"Finished {date_str}")

        except Exception as e:

            print(f"Skipped {date_str}: {e}")

    full_season = pd.concat(all_games, ignore_index=True)

    full_season.to_csv('full_season.csv', index = False)

    return full_season

In [5]:
start = datetime(2025, 11, 3)
end   = datetime.today()

season_df = build_full_season(start, end)

Found 169 game(s) on 2025-11-03
Retrieved 401826083 (1/169)
Retrieved 401824809 (2/169)
Retrieved 401826885 (3/169)
Retrieved 401812785 (4/169)
Retrieved 401820577 (5/169)
Retrieved 401817239 (6/169)
Retrieved 401819834 (7/169)
Retrieved 401813756 (8/169)
Retrieved 401826784 (9/169)
Retrieved 401812260 (10/169)
Retrieved 401827170 (11/169)
Retrieved 401823483 (12/169)
Retrieved 401823474 (13/169)
Retrieved 401812282 (14/169)
Retrieved 401819882 (15/169)
Retrieved 401824031 (16/169)
Retrieved 401827219 (17/169)
Retrieved 401823449 (18/169)
Retrieved 401826911 (19/169)
Retrieved 401817262 (20/169)
Retrieved 401817194 (21/169)
Retrieved 401830655 (22/169)
Retrieved 401827039 (23/169)
Retrieved 401828511 (24/169)
Retrieved 401826859 (25/169)
Retrieved 401829447 (26/169)
Retrieved 401825586 (27/169)
Retrieved 401827511 (28/169)
Retrieved 401824943 (29/169)
Retrieved 401823513 (30/169)
Retrieved 401827734 (31/169)
Retrieved 401827274 (32/169)
Retrieved 401824867 (33/169)
Retrieved 401823374 

In [6]:
season_df.to_csv('master_df.csv', index=False)